## Flight delays in TunisAir

This notebook is a response to the [flight delay prediction challenge in Zindi](https://zindi.world/competitions/flight-delay-prediction-challenge).
The challenge provides a training dataset including 10 variables. These contain nine predictors with information of the flights of the airline across a three-year span. The target variable is the flight delay (in minutes).

## Exploratory data analysis

We converted the date information into pythons datetime format and use it to create additional variables describing time ("hour", "weekday", "month", "year"). Spaces in variables aircraft (AC) and flight (FLTID) were removed.
The hour variable was re-classified into four time periods ("Morning", "Afternoon", "Evening", "Night") meaningful for airline operators. Similarly, the weekdays were reclassified into one delay-related variable "weekend" (True: Friday, Saturday, Sunday, Monday and False: Tuesday, Wednesday, Thursday).

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
#from data_profiling import ProfileReport
import numpy as np
import datetime as dt

#Loading data
df = pd.read_csv('data/raw/Train.csv')

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
df.info()


Number of rows: 107833
Number of columns: 10
<class 'pandas.DataFrame'>
RangeIndex: 107833 entries, 0 to 107832
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      107833 non-null  str    
 1   DATOP   107833 non-null  str    
 2   FLTID   107833 non-null  str    
 3   DEPSTN  107833 non-null  str    
 4   ARRSTN  107833 non-null  str    
 5   STD     107833 non-null  str    
 6   STA     107833 non-null  str    
 7   STATUS  107833 non-null  str    
 8   AC      107833 non-null  str    
 9   target  107833 non-null  float64
dtypes: float64(1), str(9)
memory usage: 8.2 MB


In [20]:
# Quick summary of categorical variables
categorical_columns = ["FLTID","DEPSTN","ARRSTN","STATUS","AC"]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts().head(5))


--- FLTID ---
Unique values: 1861
FLTID
WKL 0000     3105
TU 0613      1284
TU 0397      1184
AOG 0000     1103
TU 0634      1075
Name: count, dtype: int64

--- DEPSTN ---
Unique values: 132
DEPSTN
TUN    42522
DJE    10252
ORY     6755
MIR     5248
MRS     2845
Name: count, dtype: int64

--- ARRSTN ---
Unique values: 128
ARRSTN
TUN    42572
DJE    10198
ORY     6755
MIR     5251
MRS     2845
Name: count, dtype: int64

--- STATUS ---
Unique values: 5
STATUS
ATA    93679
SCH    13242
DEP      467
RTR      294
DEL      151
Name: count, dtype: int64

--- AC ---
Unique values: 68
AC
TU 320IMU    4724
TU 320IMW    4685
TU 320IMT    4634
TU 320IMR    4601
TU 320IMV    4589
Name: count, dtype: int64


In [21]:
# Finding leading/trailing whitespace in text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    original = df[column].astype(str)
    cleaned = original.str.strip()

    whitespace_rows = (original != cleaned).sum()

    print(f"{column}: {whitespace_rows} rows with leading/trailing spaces")

#print("Unique FLTID before cleaning:", df["FLTID"].nunique())

ID: 0 rows with leading/trailing spaces
DATOP: 0 rows with leading/trailing spaces
FLTID: 107830 rows with leading/trailing spaces
DEPSTN: 0 rows with leading/trailing spaces
ARRSTN: 0 rows with leading/trailing spaces
STD: 0 rows with leading/trailing spaces
STA: 0 rows with leading/trailing spaces
STATUS: 0 rows with leading/trailing spaces
AC: 0 rows with leading/trailing spaces


In [22]:
df_clean = df.copy()

# Removing leading/trailing whitespace from text columns
df_clean["FLTID"] = df_clean["FLTID"].str.strip()
print("Unique FLTID after cleaning:", df_clean["FLTID"].nunique())

Unique FLTID after cleaning: 1861


In [23]:
df_clean["FLTID"] = df["FLTID"].str.strip()
df["FLTID"] = df["FLTID"].replace(" ", "", regex=True)
df["AC"] = df["AC"].replace(" ", "", regex=True)

In [24]:
df_clean["DATOP"] = pd.to_datetime(df_clean["DATOP"],format="%Y-%m-%d")
#df_clean["DATOP"].head()
#df_clean["DATOP"].dtype

df_clean["STD"] = pd.to_datetime(df_clean["STD"],format="%Y-%m-%d %H:%M:%S") # STD using : as separator
df_clean["STD"].dtype

#STA using . as separator, this line will convert it to datetime format same as STD and DATOP
df_clean["STA"] = pd.to_datetime(df_clean["STA"], format="%Y-%m-%d %H.%M.%S") 
df_clean["STA"].dtype

dtype('<M8[us]')

## Importing Airport data

In [25]:
import airportsdata
from zoneinfo import ZoneInfo
import datetime as dt

# Load airport data by IATA code
airports = airportsdata.load("IATA")

# Add timezone based on departure and arrival airport
df_clean["DEP_timezone"] = df_clean["DEPSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

df_clean["ARR_timezone"] = df_clean["ARRSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

#print(df_clean["DEP_timezone"].isna().sum())
#print(df_clean["ARR_timezone"].isna().sum())

print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())

# Historical airport (SXF) not found in current airportsdata
df_clean.loc[df_clean["DEPSTN"] == "SXF", "DEP_timezone"] = "Europe/Berlin"
df_clean.loc[df_clean["ARRSTN"] == "SXF", "ARR_timezone"] = "Europe/Berlin"

df_clean.columns

<StringArray>
['SXF']
Length: 1, dtype: str
<StringArray>
['SXF']
Length: 1, dtype: str


Index(['ID', 'DATOP', 'FLTID', 'DEPSTN', 'ARRSTN', 'STD', 'STA', 'STATUS',
       'AC', 'target', 'DEP_timezone', 'ARR_timezone'],
      dtype='str')

In [26]:
print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())
df_clean.info()


<StringArray>
[]
Length: 0, dtype: str
<StringArray>
[]
Length: 0, dtype: str
<class 'pandas.DataFrame'>
RangeIndex: 107833 entries, 0 to 107832
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   ID            107833 non-null  str           
 1   DATOP         107833 non-null  datetime64[us]
 2   FLTID         107833 non-null  str           
 3   DEPSTN        107833 non-null  str           
 4   ARRSTN        107833 non-null  str           
 5   STD           107833 non-null  datetime64[us]
 6   STA           107833 non-null  datetime64[us]
 7   STATUS        107833 non-null  str           
 8   AC            107833 non-null  str           
 9   target        107833 non-null  float64       
 10  DEP_timezone  107833 non-null  str           
 11  ARR_timezone  107833 non-null  str           
dtypes: datetime64[us](3), float64(1), str(8)
memory usage: 9.9 MB


### Converting to UTC

In [27]:
def to_utc(dt, tz):
    if pd.isna(dt) or pd.isna(tz):
        return pd.NaT
    
    return (
        dt.replace(tzinfo=ZoneInfo(tz))
          .astimezone(ZoneInfo("UTC"))
    )


df_clean["STD_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STD"],
        df_clean["DEP_timezone"]
    )
]

df_clean["STA_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STA"],
        df_clean["ARR_timezone"]
    )
]



### Calculate flight duration in minutes

In [28]:
df_clean["duration"] = (df_clean["STA_UTC"] - df_clean["STD_UTC"]).dt.total_seconds() / 60

print(df_clean["duration"].isna().sum())

0


### Weekday and season features

In [29]:
df_clean["weekday"] = df_clean["STD_UTC"].dt.dayofweek
df_clean["weekend"] = df_clean["weekday"].map({0:True, 1:False, 2:False, 3:False, 4:True, 5:True, 6:True})
df_clean["season"] = df_clean["STD_UTC"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})


In [30]:
df_clean[df_clean["STA"].dt.year == 2019].shape[0]
df_clean['hour'] = df_clean["STD_UTC"].dt.hour
df_clean['weekday'] = df_clean["STD_UTC"].dt.weekday
df_clean['month'] = df_clean["STD_UTC"].dt.month
df_clean['year'] = df_clean["STD_UTC"].dt.year

 ### Adding the route

In [31]:
df_clean["route"] = (df_clean["DEPSTN"] + "_" + df_clean["ARRSTN"])

Adding Rotation sequence

In [32]:
df_clean = df_clean.sort_values(["DATOP","AC"]).copy()

df_clean["rotation_sequence"] = (df_clean.groupby(["DATOP", "AC"]).cumcount() + 1)

df_clean.head(10)

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target,...,STA_UTC,duration,weekday,weekend,season,hour,month,year,route,rotation_sequence
6928,train_id_6928,2016-01-01,TU 0635,ORY,DJE,2016-01-01 17:15:00,2016-01-01 20:00:00,ATA,TU 31AIMJ,18.0,...,2016-01-01 19:00:00+00:00,165.0,4,True,Winter,16,1,2016,ORY_DJE,1
7186,train_id_7186,2016-01-01,TU 0634,DJE,ORY,2016-01-01 13:35:00,2016-01-01 16:15:00,ATA,TU 31AIMJ,8.0,...,2016-01-01 15:15:00+00:00,160.0,4,True,Winter,12,1,2016,DJE_ORY,2
858,train_id_858,2016-01-01,TU 0753,FCO,TUN,2016-01-01 10:05:00,2016-01-01 11:20:00,ATA,TU 31BIMO,5.0,...,2016-01-01 10:20:00+00:00,75.0,4,True,Winter,9,1,2016,FCO_TUN,1
4511,train_id_4511,2016-01-01,TU 0752,TUN,FCO,2016-01-01 07:40:00,2016-01-01 09:00:00,ATA,TU 31BIMO,12.0,...,2016-01-01 08:00:00+00:00,80.0,4,True,Winter,6,1,2016,TUN_FCO,2
4520,train_id_4520,2016-01-01,TU 0472,TUN,VCE,2016-01-01 13:00:00,2016-01-01 14:40:00,ATA,TU 31BIMO,22.0,...,2016-01-01 13:40:00+00:00,100.0,4,True,Winter,12,1,2016,TUN_VCE,3
15117,train_id_15117,2016-01-01,TU 0473,VCE,TUN,2016-01-01 15:35:00,2016-01-01 17:20:00,ATA,TU 31BIMO,0.0,...,2016-01-01 16:20:00+00:00,105.0,4,True,Winter,14,1,2016,VCE_TUN,4
4512,train_id_4512,2016-01-01,TU 0856,TUN,MXP,2016-01-01 07:35:00,2016-01-01 09:20:00,ATA,TU 31BIMQ,25.0,...,2016-01-01 08:20:00+00:00,105.0,4,True,Winter,6,1,2016,TUN_MXP,1
4513,train_id_4513,2016-01-01,TU 0857,MXP,TUN,2016-01-01 10:15:00,2016-01-01 12:10:00,ATA,TU 31BIMQ,27.0,...,2016-01-01 11:10:00+00:00,115.0,4,True,Winter,9,1,2016,MXP_TUN,2
6917,train_id_6917,2016-01-01,TU 0853,FCO,TUN,2016-01-01 16:05:00,2016-01-01 17:20:00,ATA,TU 31BIMQ,0.0,...,2016-01-01 16:20:00+00:00,75.0,4,True,Winter,15,1,2016,FCO_TUN,3
7185,train_id_7185,2016-01-01,TU 0852,TUN,FCO,2016-01-01 13:40:00,2016-01-01 15:05:00,ATA,TU 31BIMQ,0.0,...,2016-01-01 14:05:00+00:00,85.0,4,True,Winter,12,1,2016,TUN_FCO,4


We used the estimated departure and arrival times to calculate the estimated duration of flights ("duration"). 

In [33]:
# Interval between previous flight and next flight
df_clean = (
    df_clean
    .sort_values(["AC", "STD_UTC"])
    .copy()
)

df_clean["previous_STA_UTC"] = (
    df_clean
    .groupby("AC")["STA_UTC"]
    .shift(1)
)
df_clean["interval"] = (
    df_clean["STD_UTC"] - df_clean["previous_STA_UTC"]
).dt.total_seconds() / 60

## Additional features
We used available information to create a new variable describing the size of the destination airports as, due to operational bottlenecks, the size of an airport might be related to flight delays.

In [34]:
# downloaded Airport size data from https://datacatalog.worldbank.org/search/dataset/0038117/global-airports
airports = pd.read_csv("data/airport_volume_airport_locations.csv")
airports = airports[["Orig", "TotalSeats"]]

# Three airports are missing from this database: 'BYJ', 'GAF', 'EBM'
# These are small airports, so we will fill them with the minimum of the other airports
missing_airports = pd.DataFrame({
    "Orig": ["BYJ", "GAF", "EBM"],
    "TotalSeats": [airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25)]})
airports = pd.concat([airports, missing_airports], ignore_index=True)

df_clean = pd.merge(df_clean, airports, how="left", left_on="DEPSTN", right_on="Orig")
df_clean.rename(columns={'TotalSeats': 'DEPSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)



In [35]:
df_clean = pd.merge(df_clean, airports, how="left", left_on="ARRSTN", right_on="Orig", suffixes=('_y', '_x'))
df_clean.rename(columns={'TotalSeats': 'ARRSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)
df_clean.head(5)

,ID,DATOP,FLTID,DEPSTN,ARRSTN,STD,STA,STATUS,AC,target,...,season,hour,month,year,route,rotation_sequence,previous_STA_UTC,interval,DEPSIZE,ARRSIZE
0,train_id_20112,2016-08-19,TU 9601,LIS,TUN,2016-08-19 04:00:00,2016-08-19 06:40:00,ATA,5K 343TQY,0.0,...,Summer,3,8,2016,LIS_TUN,3,NaT,NaN,1.665133e+07,4.112819e+06
1,train_id_836,2016-08-19,TU 0788,TUN,BRU,2016-08-19 07:30:00,2016-08-19 10:05:00,ATA,5K 343TQY,46.0,...,Summer,6,8,2016,TUN_BRU,1,2016-08-19 05:40:00+00:00,50.0,4.112819e+06,1.678180e+07
2,train_id_8187,2016-08-19,TU 0789,BRU,TUN,2016-08-19 11:35:00,2016-08-19 14:10:00,ATA,5K 343TQY,35.0,...,Summer,9,8,2016,BRU_TUN,2,2016-08-19 08:05:00+00:00,90.0,1.678180e+07,4.112819e+06
3,train_id_1353,2016-08-20,TU 0788,TUN,BRU,2016-08-20 07:30:00,2016-08-20 10:05:00,ATA,5K 343TQY,38.0,...,Summer,6,8,2016,TUN_BRU,1,2016-08-19 13:10:00+00:00,1040.0,4.112819e+06,1.678180e+07
4,train_id_13557,2016-08-20,TU 0789,BRU,TUN,2016-08-20 11:35:00,2016-08-20 14:15:00,ATA,5K 343TQY,16.0,...,Summer,9,8,2016,BRU_TUN,2,2016-08-20 08:05:00+00:00,90.0,1.678180e+07,4.112819e+06


## Saving clean version (Ready to train)

In [36]:
df_clean.to_csv("data/processed/Train_cleaned.csv", index=False)